In [1]:
!pip -q install sentence-transformers rank-bm25 gradio transformers sentencepiece

In [2]:
import re
import numpy as np
import gradio as gr

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from transformers import pipeline

In [3]:
documents = [
    {
        "title": "Introduction to RAG",
        "text": """
        Retrieval-Augmented Generation, or RAG, combines information
        retrieval with a language model. It retrieves relevant passages
        from a knowledge base and uses them to answer questions.
        RAG can reduce unsupported answers by grounding responses in
        retrieved information.
        """
    },
    {
        "title": "Text Chunking",
        "text": """
        Text chunking divides large documents into smaller passages.
        Chunk size affects retrieval quality. Very small chunks may
        lose context, while very large chunks may contain irrelevant
        information. Overlapping chunks can preserve context between
        neighboring passages.
        """
    },
    {
        "title": "Semantic Search",
        "text": """
        Semantic search uses embeddings to represent the meaning of text.
        It compares the vector representation of a question with document
        vectors. Semantic search can find relevant passages even when
        the question uses different words from the document.
        """
    },
    {
        "title": "Keyword Search",
        "text": """
        Keyword search finds documents containing matching terms.
        BM25 is a ranking algorithm commonly used for keyword retrieval.
        It can be particularly useful for names, technical terms,
        identifiers, and exact phrases.
        """
    },
    {
        "title": "Hybrid Search and Reranking",
        "text": """
        Hybrid search combines semantic search and keyword search.
        Reciprocal Rank Fusion can combine their ranked results.
        A cross-encoder reranker then scores each query and passage
        together to improve relevance. Only the best passages should
        be passed to the language model.
        """
    },
    {
        "title": "RAG Evaluation",
        "text": """
        RAG systems can be evaluated by checking retrieval relevance,
        answer correctness, faithfulness to source documents, and
        whether the system abstains when the required information
        is unavailable.
        """
    }
]

print("Total documents:", len(documents))

for doc in documents:
    print("-", doc["title"])

Total documents: 6
- Introduction to RAG
- Text Chunking
- Semantic Search
- Keyword Search
- Hybrid Search and Reranking
- RAG Evaluation


In [4]:
def chunk_text(text, chunk_size=45, overlap=10):
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))

        if end >= len(words):
            break

        start = end - overlap

    return chunks


chunks = []

for doc in documents:
    for i, chunk in enumerate(chunk_text(doc["text"])):
        chunks.append({
            "title": doc["title"],
            "chunk_id": i,
            "text": chunk
        })

print("Total chunks created:", len(chunks))
print("\nExample chunk:")
print(chunks[0])

Total chunks created: 6

Example chunk:
{'title': 'Introduction to RAG', 'chunk_id': 0, 'text': 'Retrieval-Augmented Generation, or RAG, combines information retrieval with a language model. It retrieves relevant passages from a knowledge base and uses them to answer questions. RAG can reduce unsupported answers by grounding responses in retrieved information.'}


In [5]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding matrix shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding matrix shape: (6, 384)


In [6]:
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())


tokenized_chunks = [
    tokenize(chunk["text"]) for chunk in chunks
]

bm25 = BM25Okapi(tokenized_chunks)

print("BM25 keyword search is ready.")

BM25 keyword search is ready.


In [7]:
def hybrid_search(query, top_k=6, rrf_k=60):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    semantic_scores = embeddings @ query_embedding
    semantic_order = np.argsort(semantic_scores)[::-1]

    keyword_scores = bm25.get_scores(tokenize(query))
    keyword_order = np.argsort(keyword_scores)[::-1]

    rrf_scores = {}

    for rank, idx in enumerate(semantic_order[:top_k * 2], start=1):
        rrf_scores[int(idx)] = (
            rrf_scores.get(int(idx), 0) + 1 / (rrf_k + rank)
        )

    for rank, idx in enumerate(keyword_order[:top_k * 2], start=1):
        rrf_scores[int(idx)] = (
            rrf_scores.get(int(idx), 0) + 1 / (rrf_k + rank)
        )

    ranked_indices = sorted(
        rrf_scores,
        key=rrf_scores.get,
        reverse=True
    )[:top_k]

    results = []

    for idx in ranked_indices:
        results.append({
            **chunks[idx],
            "rrf_score": rrf_scores[idx]
        })

    return results


results = hybrid_search(
    "How does hybrid search improve retrieval?"
)

for result in results:
    print(result["title"], ":", result["text"])
    print("RRF score:", round(result["rrf_score"], 5))
    print("-" * 50)

Hybrid Search and Reranking : Hybrid search combines semantic search and keyword search. Reciprocal Rank Fusion can combine their ranked results. A cross-encoder reranker then scores each query and passage together to improve relevance. Only the best passages should be passed to the language model.
RRF score: 0.03279
--------------------------------------------------
Introduction to RAG : Retrieval-Augmented Generation, or RAG, combines information retrieval with a language model. It retrieves relevant passages from a knowledge base and uses them to answer questions. RAG can reduce unsupported answers by grounding responses in retrieved information.
RRF score: 0.032
--------------------------------------------------
RAG Evaluation : RAG systems can be evaluated by checking retrieval relevance, answer correctness, faithfulness to source documents, and whether the system abstains when the required information is unavailable.
RRF score: 0.0315
---------------------------------------------

In [8]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)


def rerank_results(query, results, top_k=3):
    if not results:
        return []

    pairs = [
        (query, result["text"])
        for result in results
    ]

    scores = reranker.predict(pairs)

    reranked = []

    for result, score in zip(results, scores):
        item = result.copy()
        item["rerank_score"] = float(score)
        reranked.append(item)

    reranked.sort(
        key=lambda item: item["rerank_score"],
        reverse=True
    )

    return reranked[:top_k]


reranked_results = rerank_results(
    "How does hybrid search improve retrieval?",
    results
)

for result in reranked_results:
    print("Title:", result["title"])
    print("Score:", round(result["rerank_score"], 4))
    print("Text:", result["text"])
    print("-" * 50)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Title: Hybrid Search and Reranking
Score: 5.0675
Text: Hybrid search combines semantic search and keyword search. Reciprocal Rank Fusion can combine their ranked results. A cross-encoder reranker then scores each query and passage together to improve relevance. Only the best passages should be passed to the language model.
--------------------------------------------------
Title: Introduction to RAG
Score: -2.2226
Text: Retrieval-Augmented Generation, or RAG, combines information retrieval with a language model. It retrieves relevant passages from a knowledge base and uses them to answer questions. RAG can reduce unsupported answers by grounding responses in retrieved information.
--------------------------------------------------
Title: Keyword Search
Score: -6.0699
Text: Keyword search finds documents containing matching terms. BM25 is a ranking algorithm commonly used for keyword retrieval. It can be particularly useful for names, technical terms, identifiers, and exact phrases.
---

In [11]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = 0 if torch.cuda.is_available() else -1

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device
)

print("Answer generation model is ready.")

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CohereCompassForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM',

Answer generation model is ready.


In [12]:
def answer_question(query):
    retrieved = hybrid_search(query, top_k=6)

    reranked = rerank_results(
        query, retrieved, top_k=3
    )

    if not reranked:
        return (
            "The information is not available in the documents.",
            "No relevant passages were retrieved.",
            "No results."
        )

    # Conservative fallback for clearly irrelevant results.
    best_score = reranked[0]["rerank_score"]

    if best_score < -1.0:
        return (
            "The information is not available in the documents.",
            "The retrieved passages were not sufficiently relevant.",
            "No sufficiently relevant passages."
        )

    context = "\n\n".join(
        f"Source: {item['title']}\n{item['text']}"
        for item in reranked
    )

    prompt = f"""
    Answer the question using only the context below.
    If the answer is not supported by the context, say that
    the information is not available in the documents.
    Do not invent facts.

    Context:
    {context}

    Question: {query}

    Answer:
    """

    output = generator(
        prompt,
        max_new_tokens=120,
        do_sample=False
    )[0]["generated_text"]

    sources = "\n\n".join(
        f"• {item['title']}: {item['text']}"
        for item in reranked
    )

    retrieval_log = "\n\n".join(
        f"{i}. {item['title']} | "
        f"Reranking score: {item['rerank_score']:.4f}"
        for i, item in enumerate(reranked, start=1)
    )

    return output, sources, retrieval_log


answer, sources, log = answer_question(
    "What is the purpose of reranking in RAG?"
)

print("ANSWER:\n", answer)
print("\nRETRIEVED SOURCES:\n", sources)
print("\nRERANKING LOG:\n", log)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER:
 
    Answer the question using only the context below.
    If the answer is not supported by the context, say that
    the information is not available in the documents.
    Do not invent facts.

    Context:
    Source: Introduction to RAG
Retrieval-Augmented Generation, or RAG, combines information retrieval with a language model. It retrieves relevant passages from a knowledge base and uses them to answer questions. RAG can reduce unsupported answers by grounding responses in retrieved information.

Source: RAG Evaluation
RAG systems can be evaluated by checking retrieval relevance, answer correctness, faithfulness to source documents, and whether the system abstains when the required information is unavailable.

Source: Hybrid Search and Reranking
Hybrid search combines semantic search and keyword search. Reciprocal Rank Fusion can combine their ranked results. A cross-encoder reranker then scores each query and passage together to improve relevance. Only the best passages

In [13]:
def rag_interface(question):
    if not question.strip():
        return "Please enter a question.", "", ""

    answer, sources, retrieval_log = answer_question(question)

    return answer, sources, retrieval_log


demo = gr.Interface(
    fn=rag_interface,
    inputs=gr.Textbox(
        label="Ask a question about the documents",
        placeholder="How does semantic search work?"
    ),
    outputs=[
        gr.Textbox(label="Generated Answer"),
        gr.Textbox(label="Retrieved Supporting Documents"),
        gr.Textbox(label="Reranking Results")
    ],
    title="Advanced RAG with Hybrid Search and Reranking",
    description=(
        "A document question-answering system using "
        "semantic search, BM25 keyword search, RRF, "
        "and cross-encoder reranking."
    )
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5dd06e5a02c8ce7a93.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [14]:
def rag_interface(question):
    if not question.strip():
        return "Please enter a question.", "", ""

    answer, sources, retrieval_log = answer_question(question)

    return answer, sources, retrieval_log


demo = gr.Interface(
    fn=rag_interface,
    inputs=gr.Textbox(
        label="Ask a question about the documents",
        placeholder="How does semantic search work?"
    ),
    outputs=[
        gr.Textbox(label="Generated Answer"),
        gr.Textbox(label="Retrieved Supporting Documents"),
        gr.Textbox(label="Reranking Results")
    ],
    title="Advanced RAG with Hybrid Search and Reranking",
    description=(
        "A document question-answering system using "
        "semantic search, BM25 keyword search, RRF, "
        "and cross-encoder reranking."
    )
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4a0886e9084bd3f913.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [15]:
def rag_interface(question):
    if not question.strip():
        return "Please enter a question.", "", ""

    answer, sources, retrieval_log = answer_question(question)

    return answer, sources, retrieval_log


demo = gr.Interface(
    fn=rag_interface,
    inputs=gr.Textbox(
        label="Ask a question about the documents",
        placeholder="How does semantic search work?"
    ),
    outputs=[
        gr.Textbox(label="Generated Answer"),
        gr.Textbox(label="Retrieved Supporting Documents"),
        gr.Textbox(label="Reranking Results")
    ],
    title="Advanced RAG with Hybrid Search and Reranking",
    description=(
        "A document question-answering system using "
        "semantic search, BM25 keyword search, RRF, "
        "and cross-encoder reranking."
    )
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://20fd9a0933913c7b6f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
